# Evaluation
## Introduction

This file is introduced for unifying evaluate the result from each pipeline, as the result has been required for recorded in standard pattern. In each record of the result from each pipeline, it will contain 3 main parts for evaluation:
1. gold_label
2. predicted_label
3. reason

The predicted labels will be introduced for evaluating whether matches the gold labels. Based on the result, the confusion matrix could be generated and then compute the matrics such as Recall, Presion and Macro-F1 for quantifying the performance. Here is an example of record from NoRAG result over training set.

```json
  {
    "claim_id": "claim_003055",
    "claim": "Kenyans working at Mara North private wildlife park have been made to carry white tourists in sedan chairs depicting modern 'slavery'",
    "gold_label": "Refuted",
    "predicted_label": "Refuted",
    "reason": "There is no widely reported or verified evidence suggesting that Kenyans are being forced to carry white tourists in sedan chairs at the Mara North private wildlife park. Such practices, if true, would likely be a significant human rights issue and have garnered attention from media and human rights organizations.",
    "model_name": "qwen2.5:7b",
    "error": null
  }
```

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    
)

In [ ]:
RESULT_PATH = Path(r"TestRAG\test_RAG_qwen3_30b.json")

LABELS = [
    "Supported",
    "Refuted",
    "Not Enough Evidence",
    "Conflicting Evidence/Cherrypicking",
]

LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

In [ ]:
def load_prediction_json(path: Path) -> pd.DataFrame:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Prediction file should be a JSON list of records.")
    df = pd.DataFrame(data)
    return df


df_raw = load_prediction_json(RESULT_PATH)
print(f"Loaded records: {len(df_raw):,}")
df_raw.head()

Below is the function to transvert the record into dataframe.

In [ ]:
def prepare_eval_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "gold_label" not in df.columns and "golden_label" in df.columns:
        df = df.rename(columns={"golden_label": "gold_label"})

    required_cols = ["claim_id", "claim", "gold_label", "predicted_label"]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    if "error" not in df.columns:
        df["error"] = None
    return df

df = prepare_eval_dataframe(df_raw)
df[["claim_id", "gold_label", "predicted_label", "error"]].head()

For better evaluation fairness, I introduced the invalid records check, if the gold label/prediceed_label is out of the valid set, or the error in the record is not Null, it will be collected as invalid records, and will not be applied in the final evaluation. 

In [ ]:
eval_df = df[
    df["gold_label"].isin(LABELS) &
    df["predicted_label"].isin(LABELS) &
    df["error"].isnull()
].copy()

print(f"Valid records for label evaluation: {len(eval_df):,}")
print(f"Dropped records: {len(df) - len(eval_df):,}")

eval_df[["claim_id", "gold_label", "predicted_label"]].head()

In [ ]:
y_true = eval_df["gold_label"].tolist()
y_pred = eval_df["predicted_label"].tolist()

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Gold: {label}" for label in LABELS],
    columns=[f"Pred: {label}" for label in LABELS],
)

cm_df

In [ ]:
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(9, 7))

    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Gold label")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))

    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center"
            )

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(cm, LABELS, title=None)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

report_dict = classification_report(
    y_true,
    y_pred,
    labels=LABELS,
    target_names=LABELS,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).T

print(f"Accuracy: {accuracy:.4f}")

report_df

In [ ]:
# from transformers import pipeline

# NLI_MODEL = "facebook/bart-large-mnli"

# nli_pipeline = pipeline(
#     "text-classification",
#     model=NLI_MODEL,
#     tokenizer=NLI_MODEL,
#     top_k=None,
#     truncation=True,
# )

In [ ]:
# def normalize_nli_label(label: str) -> str:
#     label = label.lower()

#     if "entail" in label:
#         return "entailment"
#     if "contrad" in label:
#         return "contradiction"
#     if "neutral" in label:
#         return "neutral"

#     return label


# def run_nli(premise: str, hypothesis: str) -> dict:
#     premise = str(premise)
#     hypothesis = str(hypothesis)

#     output = nli_pipeline({
#         "text": premise,
#         "text_pair": hypothesis,
#     })
#     if len(output) > 0 and isinstance(output[0], list):
#         scores = output[0]
#     else:
#         scores = output

#     score_dict = {
#         normalize_nli_label(item["label"]): item["score"]
#         for item in scores
#     }

#     predicted_relation = max(score_dict, key=score_dict.get)

#     return {
#         "nli_relation": predicted_relation,
#         "nli_entailment_score": score_dict.get("entailment", 0.0),
#         "nli_contradiction_score": score_dict.get("contradiction", 0.0),
#         "nli_neutral_score": score_dict.get("neutral", 0.0),
#     }

In [ ]:
# EXPECTED_NLI_RELATION = {
#     "Supported": "entailment",
#     "Refuted": "contradiction",
#     "Not Enough Evidence": "neutral",
#     "Conflicting Evidence/Cherrypicking": None,
# }

In [ ]:
# sample_df = eval_df.head(20).copy()

# nli_results = []

# for _, row in sample_df.iterrows():
#     result = run_nli(
#         premise=row["reason"],
#         hypothesis=row["claim"],
#     )
#     nli_results.append(result)

# nli_sample_df = pd.concat(
#     [
#         sample_df.reset_index(drop=True),
#         pd.DataFrame(nli_results),
#     ],
#     axis=1,
# )

# nli_sample_df["expected_nli_relation"] = nli_sample_df["predicted_label"].map(EXPECTED_NLI_RELATION)

# nli_sample_df["reason_label_consistent"] = (
#     nli_sample_df["nli_relation"] == nli_sample_df["expected_nli_relation"]
# )

# nli_sample_df[[
#     "claim_id",
#     "claim",
#     "predicted_label",
#     "reason",
#     "nli_relation",
#     "expected_nli_relation",
#     "reason_label_consistent",
# ]]

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


FILES = {
    "200/50": Path(r"output\CrossEncoder\Hybrid\word_200_50_retrieval_top10_rerank_True_Dedup_True_qwen3_30b_result.json"),
    "100/25": Path(r"output\CrossEncoder\Hybrid\word_100_25_retrieval_top10_rerank_True_Dedup_True_qwen3_30b_result.json"),
    "Sentence": Path(r"output\CrossEncoder\Hybrid\sentence_retrieval_top10_rerank_True_Dedup_True_qwen3_30b_result.json"),
}

REFERENCE = "Sentence"
N_BOOTSTRAP = 1000
SEED = 42

LABELS = [
    "Supported",
    "Refuted",
    "Not Enough Evidence",
    "Conflicting Evidence/Cherrypicking",
]


def load_result(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    return df[["claim_id", "gold_label", "predicted_label"]].sort_values("claim_id")


def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0,
        ),
        "Supported F1": f1_score(
            y_true, y_pred,
            labels=["Supported"],
            average="macro",
            zero_division=0,
        ),
        "Refuted F1": f1_score(
            y_true, y_pred,
            labels=["Refuted"],
            average="macro",
            zero_division=0,
        ),
    }


results = {name: load_result(path) for name, path in FILES.items()}
reference_df = results[REFERENCE]

gold = reference_df["gold_label"].to_numpy()
reference_pred = reference_df["predicted_label"].to_numpy()

# 每个类别在原始development set中的位置
class_indices = [
    np.where(gold == label)[0]
    for label in LABELS
]

rng = np.random.default_rng(SEED)
output = []

for name, alternative_df in results.items():

    if name == REFERENCE:
        continue

    # 检查两个结果是否对应相同claims
    assert np.array_equal(
        reference_df["claim_id"].to_numpy(),
        alternative_df["claim_id"].to_numpy(),
    )

    assert np.array_equal(
        gold,
        alternative_df["gold_label"].to_numpy(),
    )

    alternative_pred = alternative_df["predicted_label"].to_numpy()

    reference_scores = calculate_metrics(gold, reference_pred)
    alternative_scores = calculate_metrics(gold, alternative_pred)

    bootstrap_differences = {
        metric: []
        for metric in reference_scores
    }

    for _ in range(N_BOOTSTRAP):

        # 在每个gold class内部有放回抽样
        sampled_indices = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True)
            for indices in class_indices
        ])

        sampled_gold = gold[sampled_indices]

        sampled_reference_scores = calculate_metrics(
            sampled_gold,
            reference_pred[sampled_indices],
        )

        sampled_alternative_scores = calculate_metrics(
            sampled_gold,
            alternative_pred[sampled_indices],
        )

        for metric in reference_scores:
            difference = (
                sampled_reference_scores[metric]
                - sampled_alternative_scores[metric]
            )
            bootstrap_differences[metric].append(difference)

    for metric in reference_scores:

        lower, upper = np.percentile(
            bootstrap_differences[metric],
            [2.5, 97.5],
        )

        output.append({
            "Comparison": f"{REFERENCE} - {name}",
            "Metric": metric,
            "Reference": reference_scores[metric],
            "Alternative": alternative_scores[metric],
            "Difference": (
                reference_scores[metric]
                - alternative_scores[metric]
            ),
            "CI Lower": lower,
            "CI Upper": upper,
            "CI excludes zero": lower > 0 or upper < 0,
        })


bootstrap_df = pd.DataFrame(output)
bootstrap_df.round(3)